# 🎛️ H.O.R.N. Antigravity Audit Orchestrator
This notebook uses the **Antigravity SDK** to reason over Ableton session metadata. It integrates with your local DuckDB/LanceDB lakehouse and uses **Pydantic V2** to enforce structured DAW diagnostics.

---
### 🛠️ Step 1: Environment & Core Imports
We initialize the `.venv_fresh` environment and load the Antigravity/Pydantic suite.

In [1]:
import sys
import os
import asyncio
from typing import List, Literal, Optional
from pydantic import BaseModel, Field

# 1. Antigravity SDK
from google.antigravity import Agent, LocalAgentConfig

# 2. Sovereign Components (H.O.R.N. Stack)
from sovereign_schemas import SovereignDAWDiagnostic, PHI_ASSISTANT_CONFIG
from chat import SovereignEngine

print(f"[*] Kernel: {sys.executable}")
print("[SUCCESS] Antigravity and Sovereign components loaded.")

[*] Kernel: c:\STUDIES_BACKUP\.venv_fresh\Scripts\python.exe
[SUCCESS] Antigravity and Sovereign components loaded.


### 📊 Step 2: Initialize Sovereign Engine
This connects to your DuckDB/LanceDB databases to provide the agent with "spectral truth."

In [2]:
# Initialize the local data engine
engine = SovereignEngine()
print(f"[ENGINE] Booted in {engine.boot_ms:.2f}ms")
print(f"[DATA] {len(engine.af)} tracks indexed in DuckDB.")

IOException: IO Error: Cannot open file "C:\STUDIES_BACKUP\Legion-Jacked-Pipeline\ableton-session-intelligence\AI_Logs\sonic_core_v2.duckdb": The system cannot find the path specified.


### 🚀 Step 3: Antigravity Agent Configuration
We define the "Agentic Muzzle" using your Pydantic schemas. This forces the local Phi-3 model to output valid DAW diagnostics.

In [ ]:
import json
from jinja2 import Template
from google.antigravity import Agent, LocalAgentConfig
from sovereign_schemas import SovereignDAWDiagnostic

# 1. THE H.O.R.N. JINJA TEMPLATE (Sourced from main.py / Cell 6)
JINJA_SESSION_EXAMINER = """
System: You are an expert Ableton Live structural evaluator.
Analyze the following track metadata matrix.
You must output a single JSON object matching the requested schema layout.

Session Database Context:
{{ data_context }}

User Query: {{ query }}
"""

# 2. ANTIGRAVITY CONFIGURATION (The Phi Spot replacement)
config = LocalAgentConfig(
    model="ollama/phi3",
    response_schema=SovereignDAWDiagnostic,
    # No temperature set here; LocalAgentConfig handles it internally or via kwargs
)

# 3. CORE INFERENCE & STRUCTURED AUDIT FUNCTION
async def run_antigravity_audit(query_intent: str, use_simulated_data: bool = True):
    """
    Renders your catalog records through the JINJA template,
    muzzles the local model via Antigravity, and returns the SovereignDAWDiagnostic object.
    """
    
    # Text layout simulation for zero-latency testing
    if use_simulated_data:
        data_matrix = """
        === 73 Aligned DSP Records Discovered ===
        Track "sub" -> BPM: 129.2 | Key: E  | RMS: -11.8 | Crest: 4.0 | Sub Energy: 22.0 | Plugins: ['AudioEffectGroupDevice']
        Track "sub" -> BPM: 129.2 | Key: A# | RMS: -12.0 | Crest: 4.4 | Sub Energy: 23.2 | Plugins: ['AudioEffectGroupDevice']
        Track "FX"  -> BPM: N/A   | Key: None| RMS: -20.9 | Crest: 6.4 | Sub Energy: 0.3  | Plugins: ['Eq8', 'Compressor2', 'Limiter']
        """
    else:
        # Fetch real data from the booted engine
        try:
            raw_rows = engine.af.head(10).to_dict(orient="records")
            data_matrix = json.dumps(raw_rows, indent=2)
        except NameError:
            data_matrix = "Engine not initialized. Using simulated fallback."

    # Render the prompt
    compiler = Template(JINJA_SESSION_EXAMINER)
    prompt = compiler.render(
        data_context=data_matrix,
        query=query_intent
    )

    print(f"🔮 Processing H.O.R.N. matrix through Antigravity Agent...")
    
    async with Agent(config) as orchestrator:
        response = await orchestrator.chat(prompt)
        
        # Returns the validated SovereignDAWDiagnostic Pydantic object
        verdict = await response.structured_output()
        
        print("\n📊 STRUCTURAL DATA VERIFICATION (Field-by-Field):")
        print(f"Target Subsystem: {verdict.diagnosed_target}")
        print(f"Severity State:   {verdict.status_flag}")
        print(f"Anomalies Array:  {verdict.anomalies_found}")
        print(f"Monologue Steps:  {verdict.assistant_remediation_monologue}")
        
        return verdict

# 4. TRIGGER TEST
# await run_antigravity_audit("Evaluate sub-bass and drum track energy allocations.")

# 🔄 LangGraph & Google-GenAI SDK Multi-Turn Integration
We successfully integrated a stateful, multi-turn orchestrator using the modern `google-genai` SDK inside a LangGraph workflow (`langgraph_gemini_test.py`).
It maintains conversational history across turns and executes local repository inspection tools.

#### 📜 Verified Integration Code:
```python
"""
LangGraph & Google-GenAI SDK Integration Test
===============================================
A stateful, multi-turn orchestrator that runs the Gemini API via the modern
google-genai SDK inside a LangGraph workflow. Exposes local repo inspection tools.
"""
import os
import sys
import asyncio
from typing import Annotated, List, Literal, Dict, Any, Union
import operator
from dotenv import load_dotenv

load_dotenv()
from google import genai
from google.genai import types
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict

def list_workspace_files() -> dict:
    """List all Python files in the current folder."""
    files = [f for f in os.listdir(".") if f.endswith(".py")]
    return {"status": "success", "files": files}

def view_file_preview(filename: str) -> dict:
    """Read the first 15 lines of a python file to inspect its code."""
    if not os.path.exists(filename):
        return {"status": "error", "message": f"File '{filename}' not found."}
    try:
        with open(filename, "r", encoding="utf-8", errors="ignore") as f:
            lines = [f.readline() for _ in range(15)]
        return {"status": "success", "filename": filename, "preview": "".join(lines)}
    except Exception as e:
        return {"status": "error", "message": str(e)}

TOOLS_REGISTRY = {
    "list_workspace_files": list_workspace_files,
    "view_file_preview": view_file_preview,
}

class AgentState(TypedDict):
    messages: Annotated[List[types.Content], operator.add]

def normalize_messages(messages: List[Union[types.Content, dict, Any]]) -> List[types.Content]:
    normalized = []
    for msg in messages:
        if isinstance(msg, types.Content):
            normalized.append(msg)
        elif isinstance(msg, dict):
            role = msg.get("role", "user")
            if role == "assistant": role = "model"
            normalized.append(types.Content(
                role=role,
                parts=[types.Part.from_text(text=msg.get("content", ""))]
            ))
    return normalized

async def call_gemini_agent(state: AgentState) -> dict:
    client = genai.Client()
    contents = normalize_messages(state["messages"])
    config = types.GenerateContentConfig(
        system_instruction="You are the LangGraph-Gemini Bridge. Inspect the local codebase.",
        temperature=0.0,
        tools=[list_workspace_files, view_file_preview]
    )
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=contents,
        config=config
    )
    model_content = response.candidates[0].content
    if not model_content.role: model_content.role = "model"
    return {"messages": [model_content]}

async def execute_tools(state: AgentState) -> dict:
    last_message = state["messages"][-1]
    tool_responses = []
    for part in last_message.parts:
        if part.function_call:
            call = part.function_call
            tool_func = TOOLS_REGISTRY.get(call.name)
            result = tool_func(**call.args) if tool_func else {"status": "error"}
            tool_responses.append(types.Part.from_function_response(
                name=call.name, response={"result": result}
            ))
    return {"messages": [types.Content(role="user", parts=tool_responses)]}

def should_continue(state: AgentState) -> Literal["tools", "end"]:
    last_message = state["messages"][-1]
    if last_message.parts and any(p.function_call for p in last_message.parts):
        return "tools"
    return "end"

def build_workflow() -> StateGraph:
    workflow = StateGraph(AgentState)
    workflow.add_node("agent", call_gemini_agent)
    workflow.add_node("tools", execute_tools)
    workflow.set_entry_point("agent")
    workflow.add_conditional_edges("agent", should_continue, {"tools": "tools", "end": END})
    workflow.add_edge("tools", "agent")
    return workflow.compile()
```

#### 📊 Verified Multi-Turn Execution Transcript:
```text
🚀 [START] Stateful LangGraph & google-genai Multi-Turn Test
======================================================================

👤 User (Turn 1): 'Please list the Python files in this folder.'

🤖 Agent (Turn 1):
Here are the Python files in this folder:

*   `apply_fixes.py`
*   `ask_antigravity.py`
*   `ask_antigravity_code.py`
*   `autonomous_search_clustering.py`
*   `build_code_mining_notebook.py`
*   `build_web_intel_notebook.py`
*   `check_env.py`
*   `check_lancedb.py`
*   `check_sources.py`
*   `chris_lake_fire_test.py`
*   `code_lane_engine.py`
*   `code_lane_steps.py`
*   `code_lane_v2.py`
*   `deep_scan.py`
*   `deterministic_omni_pipeline.py`
*   `export_chat.py`
*   `find_chris_lake.py`
*   `find_interferences.py`
*   `forest_engine_cell.py`
*   `github_audit.py`
*   `langgraph_gemini_test.py`
*   `main.py`
*   `master_batch.py`
*   `music_catelog.py`
*   `patch_notebook.py`
*   `pydantic_extractor.py`
*   `rag_api.py`
*   `run_data_dump.py`
*   `sovereign_schemas.py`
*   `spider_web_mapper.py`
*   `spotify_probe.py`
*   `test_ingest.py`
*   `test_itunes.py`
*   `update_nb.py`
*   `vectorize_content.py`
*   `web_scan.py`
----------------------------------------------------------------------

👤 User (Turn 2): 'Great. Can you show me the first 15 lines of update_nb.py?'

🤖 Agent (Turn 2):
Here's a preview of the first 15 lines of `update_nb.py`:

```python
import json
import re

with open('gemini_code_lane_cells.md', 'r', encoding='utf-8', errors='ignore') as f:
    text = f.read()

# find all cell objects
# They look like { "cell_type": ... }
cells = []
matches = re.finditer(r'\{\s*"cell_type":\s*".*?\}', text, re.DOTALL)
# wait, regex for nested brackets is hard.
# Let's just find the first { "cell_type" and the last } before ```
start = text.find('{\n "cell_type"')
end = text.rfind('}')
if start != -1 and end != -1:
```
======================================================================
✅ [COMPLETE] Multi-Turn Workflow Execution Successful.
```


In [ ]:
import os
import json
import duckdb
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from typing import List
from dotenv import load_dotenv

load_dotenv()

# ─── 1. PYDANTIC READINESS SCHEMATION ───
class OperationReadiness(BaseModel):
    is_ready: bool = Field(
        description="True if all required session features, database tables, and API environment configs are present to perform the requested operation."
    )
    missing_resources: List[str] = Field(
        default_factory=list,
        description="List of missing database tables, files, or variables needed to run the operation."
    )
    remediation_steps: List[str] = Field(
        default_factory=list,
        description="Actionable steps or commands the user must run to provision the missing tables or files."
    )

# ─── 2. DYNAMIC METADATA DISCOVERY ───
def discover_pipeline_state() -> dict:
    """Discovers all available database tables and workspace files to feed into the readiness check."""
    state = {
        "db_tables": {},
        "available_files": [f for f in os.listdir(".") if f.endswith(".py") or f.endswith(".ipynb") or f.endswith(".md")]
    }
    
    # Check web_intel_sonicdb.duckdb
    db1 = "web_intel_sonicdb.duckdb"
    if os.path.exists(db1):
        try:
            conn = duckdb.connect(db1)
            state["db_tables"][db1] = [r[0] for r in conn.execute("SHOW TABLES").fetchall()]
            conn.close()
        except Exception as e:
            state["db_tables"][db1] = f"Error reading tables: {e}"
            
    # Check sonic_core_v2.duckdb
    db2 = "C:/STUDIES_BACKUP/Legion-Jacked-Pipeline/AI_Logs/sonic_core_v2.duckdb"
    if os.path.exists(db2):
        try:
            conn = duckdb.connect(db2)
            state["db_tables"]["sonic_core_v2.duckdb"] = [r[0] for r in conn.execute("SHOW TABLES").fetchall()]
            conn.close()
        except Exception as e:
            state["db_tables"]["sonic_core_v2.duckdb"] = f"Error reading tables: {e}"
            
    return state

# ─── 3. PRE-FLIGHT VERIFIER ───
def check_session_readiness(user_query: str, pipeline_state: dict) -> OperationReadiness:
    """
    Pre-flight validation using structured Gemini schema inference to check if the agent
    is ready to execute or if it requires more input parameters.
    """
    client = genai.Client()
    
    system_prompt = (
        "You are the Pre-Flight Database & File Readiness Validator.\n"
        "Your task is to compare the User's Query against the available Database Tables and Workspace Files.\n"
        "Verify if the tables or files targeted by the query actually exist in the provided state.\n"
        "If any table, database, or file referenced in the query is missing, set is_ready to False.\n"
        "Return the validated verdict using the OperationReadiness schema."
    )
    
    user_prompt = f"""
    User Query: {{user_query}}
    
    [DATABASE TABLES CURRENTLY LOADED]
    {{json.dumps(pipeline_state['db_tables'], indent=2)}}
    
    [WORKSPACE FILES AVAILABLE]
    {{json.dumps(pipeline_state['available_files'], indent=2)}}
    
    API Key Setup: {{'GEMINI_API_KEY present' if os.environ.get('GEMINI_API_KEY') else 'GEMINI_API_KEY MISSING'}}
    """
    
    response = client.models.generate_content(
        model="gemini-flash-latest", # Quota-safe 1.5 Flash endpoint
        contents=user_prompt.replace('{{user_query}}', user_query)
                            .replace('{{json.dumps(pipeline_state[\'db_tables\'], indent=2)}}', json.dumps(pipeline_state['db_tables'], indent=2))
                            .replace('{{json.dumps(pipeline_state[\'available_files\'], indent=2)}}', json.dumps(pipeline_state['available_files'], indent=2))
                            .replace("{{'GEMINI_API_KEY present' if os.environ.get('GEMINI_API_KEY') else 'GEMINI_API_KEY MISSING'}}", 'GEMINI_API_KEY present' if os.environ.get('GEMINI_API_KEY') else 'GEMINI_API_KEY MISSING'),
        config=types.GenerateContentConfig(
            system_instruction=system_prompt,
            response_mime_type="application/json",
            response_schema=OperationReadiness,
            temperature=0.0
        )
    )
    
    return OperationReadiness.model_validate_json(response.text)

# ─── 4. EXECUTE TEST RUNS ───
pipeline_state = discover_pipeline_state()
print(f"📦 Discovered Pipeline State: {json.dumps(pipeline_state, indent=2)}")
print("=" * 70)

# Query 1: Valid table query
q1 = "Get track details and dsp_rms from spotify_charts_daily."
print(f"\n🔍 Testing Query 1: '{q1}'")
verdict1 = check_session_readiness(q1, pipeline_state)
print(f"Ready? {'🟢 YES' if verdict1.is_ready else '🔴 NO'}")
if not verdict1.is_ready:
    print(f"  Missing: {verdict1.missing_resources}")
    print(f"  Remediation: {verdict1.remediation_steps}")

# Query 2: Invalid table query
q2 = "Audit soundcloud_metadata and check rhythm vectors."
print(f"\n🔍 Testing Query 2: '{q2}'")
verdict2 = check_session_readiness(q2, pipeline_state)
print(f"Ready? {'🟢 YES' if verdict2.is_ready else '🔴 NO'}")
if not verdict2.is_ready:
    print(f"  Missing: {verdict2.missing_resources}")
    print(f"  Remediation: {verdict2.remediation_steps}")


# 🎮 Band of Five: Interactive Stem Mastering RPG
Run the code cell below to initialize the Infinite Studio session. 
Work with **The Conductor, Composer, Sound Engineer, Guardian, and Trickster DJ** to master each stem step-by-step. 
Provide text feedback to adjust the virtual pedalboard settings, or type **'YES'** to approve and advance to the next stem!

In [ ]:
import sys
import os
# Ensure project root is in Python path
sys.path.insert(0, r"c:\STUDIES_BACKUP\Legion-Jacked-Pipeline")

from app.music_game import initialize_game, process_game_input

def print_studio_status(sess):
    print("=" * 80)
    print(f"🎵 INFINITE STUDIO SESSION: {sess.track_title}")
    if not sess.game_over:
        current = sess.stems[sess.current_stem_index]
        print(f"🎚️ Mastering Stem: '{current.name}' ({current.stem_type.upper()}) [{sess.current_stem_index + 1}/{len(sess.stems)}]")
    else:
        print("🏆 SESSION MASTERING COMPLETE!")
    print("=" * 80)
    
    for log in sess.stage_logs:
        print(f"\n{log['character']}: {log['message']}")
    print("-" * 80)
    if not sess.game_over:
        print("💬 Actions: Type 'YES' to approve, or type adjustments (e.g. 'boost vocals', 'cut low end').")

# 1. Initialize the game session
stems_dir = r"C:\Users\adams\Downloads\GIRL NAME DREAM -  i need that Stems"
game = initialize_game("GIRL NAME DREAM - i need that", stems_dir)
print_studio_status(game)


In [ ]:
# Run this cell repeatedly to submit your actions and see the Band's response!
# Change the action variable below to submit input:
action = "YES"

game = process_game_input(game, action)
print_studio_status(game)
